### Context Aware Music Recommender System

##### Imports

In [134]:
import pandas as pd
import numpy as np
import re
import warnings
import shap
import lime
import lime.lime_tabular
import matplotlib
matplotlib.use('Agg')           # non-interactive backend — works in Jupyter & scripts
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack, csr_matrix
warnings.filterwarnings('ignore')

##### Last.fm

### Read the datas

In [ ]:

### last.fm data
columns = ['userid', 'timestamp', 'artid', 'artname', 'traid', 'traname']

df_lastfm = pd.read_csv(
        'data/lastfm-dataset-1K/userid-timestamp-artid-artname-traid-traname.tsv', 
        sep='\t', 
        on_bad_lines='skip',
        names=columns, 
        header=None,
        nrows=1000000  # full load is 19 millions rows,
    )
df_lastfm


,userid,timestamp,artid,artname,traid,traname
0,user_000001,2009-05-04T23:08:57Z,f1b1cf71-bd35-4e99-8624-24a6e15f133a,Deep Dish,NaN,Fuck Me Im Famous (Pacha Ibiza)-09-28-2007
1,user_000001,2009-05-04T13:54:10Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Composition 0919 (Live_2009_4_15)
2,user_000001,2009-05-04T13:52:04Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc2 (Live_2009_4_15)
3,user_000001,2009-05-04T13:42:52Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Hibari (Live_2009_4_15)
4,user_000001,2009-05-04T13:42:11Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc1 (Live_2009_4_15)
...,...,...,...,...,...,...
999995,user_000041,2006-09-05T10:59:29Z,8385f26a-f374-4a04-9aad-4442ce24db2b,Vnv Nation,0ce0ad1a-2a1c-4870-9434-08adf7b1b46e,Forsaken
999996,user_000041,2006-09-05T10:54:28Z,8385f26a-f374-4a04-9aad-4442ce24db2b,Vnv Nation,0ce0ad1a-2a1c-4870-9434-08adf7b1b46e,Forsaken
999997,user_000041,2006-09-05T10:49:28Z,8385f26a-f374-4a04-9aad-4442ce24db2b,Vnv Nation,0ce0ad1a-2a1c-4870-9434-08adf7b1b46e,Forsaken
999998,user_000041,2006-09-05T10:44:27Z,8385f26a-f374-4a04-9aad-4442ce24db2b,Vnv Nation,0ce0ad1a-2a1c-4870-9434-08adf7b1b46e,Forsaken


In [136]:
df_lastfm.shape

(1000000, 6)

In [137]:
df_lastfm.columns

Index(['userid', 'timestamp', 'artid', 'artname', 'traid', 'traname'], dtype='str')

##### Spotify Data

In [138]:
df_spotify = pd.read_csv('data/spotify-dataset/data.csv')  # kaggle 
df_spotify


,valence,year,acousticness,artists,danceability,duration_ms,energy,explicit,id,instrumentalness,key,liveness,loudness,mode,name,popularity,release_date,speechiness,tempo
0,0.0594,1921,0.98200,"['Sergei Rachmaninoff', 'James Levine', 'Berli...",0.279,831667,0.211,0,4BJqT0PrAfrxzMOxytFOIz,0.878000,10,0.6650,-20.096,1,"Piano Concerto No. 3 in D Minor, Op. 30: III. ...",4,1921,0.0366,80.954
1,0.9630,1921,0.73200,['Dennis Day'],0.819,180533,0.341,0,7xPhfUan2yNtyFG0cUWkt8,0.000000,7,0.1600,-12.441,1,Clancy Lowered the Boom,5,1921,0.4150,60.936
2,0.0394,1921,0.96100,['KHP Kridhamardawa Karaton Ngayogyakarta Hadi...,0.328,500062,0.166,0,1o6I8BglA6ylDMrIELygv1,0.913000,3,0.1010,-14.850,1,Gati Bali,5,1921,0.0339,110.339
3,0.1650,1921,0.96700,['Frank Parker'],0.275,210000,0.309,0,3ftBPsC5vPBKxYSee08FDH,0.000028,5,0.3810,-9.316,1,Danny Boy,3,1921,0.0354,100.109
4,0.2530,1921,0.95700,['Phil Regan'],0.418,166693,0.193,0,4d6HGyGT8e121BsdKmw9v6,0.000002,3,0.2290,-10.096,1,When Irish Eyes Are Smiling,2,1921,0.0380,101.665
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170648,0.6080,2020,0.08460,"['Anuel AA', 'Daddy Yankee', 'KAROL G', 'Ozuna...",0.786,301714,0.808,0,0KkIkfsLEJbrcIhYsCL7L5,0.000289,7,0.0822,-3.702,1,China,72,2020-05-29,0.0881,105.029
170649,0.7340,2020,0.20600,['Ashnikko'],0.717,150654,0.753,0,0OStKKAuXlxA0fMH54Qs6E,0.000000,7,0.1010,-6.020,1,Halloweenie III: Seven Days,68,2020-10-23,0.0605,137.936
170650,0.6370,2020,0.10100,['MAMAMOO'],0.634,211280,0.858,0,4BZXVFYCb76Q0Klojq4piV,0.000009,4,0.2580,-2.226,0,AYA,76,2020-11-03,0.0809,91.688
170651,0.1950,2020,0.00998,['Eminem'],0.671,337147,0.623,1,5SiZJoLXp3WOl3J4C8IK0d,0.000008,2,0.6430,-7.161,1,Darkness,70,2020-01-17,0.3080,75.055


### Feature Extraction

In [139]:
# extract datetime
df_lastfm["datetime"] = pd.to_datetime(df_lastfm["timestamp"], utc=True)
df_lastfm.head(3)

,userid,timestamp,artid,artname,traid,traname,datetime
0,user_000001,2009-05-04T23:08:57Z,f1b1cf71-bd35-4e99-8624-24a6e15f133a,Deep Dish,NaN,Fuck Me Im Famous (Pacha Ibiza)-09-28-2007,2009-05-04 23:08:57+00:00
1,user_000001,2009-05-04T13:54:10Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Composition 0919 (Live_2009_4_15),2009-05-04 13:54:10+00:00
2,user_000001,2009-05-04T13:52:04Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc2 (Live_2009_4_15),2009-05-04 13:52:04+00:00


In [140]:
# get hour and time_of_day
def get_time_of_day(hour):
    if 5 <= hour < 12:
        return "morning"
    elif 12 <= hour < 17:
        return "afternoon"
    elif 17 <= hour < 21:
        return "evening"
    else:
        return "night"

df_lastfm["hour"] = df_lastfm["datetime"].dt.hour
df_lastfm["time_of_day"] = df_lastfm["hour"].apply(get_time_of_day)
df_lastfm.head(3)

,userid,timestamp,artid,artname,traid,traname,datetime,hour,time_of_day
0,user_000001,2009-05-04T23:08:57Z,f1b1cf71-bd35-4e99-8624-24a6e15f133a,Deep Dish,NaN,Fuck Me Im Famous (Pacha Ibiza)-09-28-2007,2009-05-04 23:08:57+00:00,23,night
1,user_000001,2009-05-04T13:54:10Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Composition 0919 (Live_2009_4_15),2009-05-04 13:54:10+00:00,13,afternoon
2,user_000001,2009-05-04T13:52:04Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc2 (Live_2009_4_15),2009-05-04 13:52:04+00:00,13,afternoon


In [141]:
# extract weekday & weekend
df_lastfm["day_of_week"] = df_lastfm["datetime"].dt.weekday
df_lastfm["day_type"] = df_lastfm["day_of_week"].apply(lambda x: "weekend" if x >= 5 else "weekday")
df_lastfm.head(3)

,userid,timestamp,artid,artname,traid,traname,datetime,hour,time_of_day,day_of_week,day_type
0,user_000001,2009-05-04T23:08:57Z,f1b1cf71-bd35-4e99-8624-24a6e15f133a,Deep Dish,NaN,Fuck Me Im Famous (Pacha Ibiza)-09-28-2007,2009-05-04 23:08:57+00:00,23,night,0,weekday
1,user_000001,2009-05-04T13:54:10Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Composition 0919 (Live_2009_4_15),2009-05-04 13:54:10+00:00,13,afternoon,0,weekday
2,user_000001,2009-05-04T13:52:04Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc2 (Live_2009_4_15),2009-05-04 13:52:04+00:00,13,afternoon,0,weekday


In [142]:
# check if weekend is extracted
df_lastfm[df_lastfm['day_type'] == 'weekend'].head()

,userid,timestamp,artid,artname,traid,traname,datetime,hour,time_of_day,day_of_week,day_type
14,user_000001,2009-05-03T15:48:25Z,ba2f4f3b-0293-4bc8-bb94-2f73b5207343,Underworld,dc394163-2b78-4b56-94e4-658597a29ef8,"Boy, Boy, Boy (Switch Remix)",2009-05-03 15:48:25+00:00,15,afternoon,6,weekend
15,user_000001,2009-05-03T15:37:56Z,ba2f4f3b-0293-4bc8-bb94-2f73b5207343,Underworld,340d9a0b-9a43-4098-b116-9f79811bd508,Crocodile (Innervisions Orchestra Mix),2009-05-03 15:37:56+00:00,15,afternoon,6,weekend
16,user_000001,2009-05-03T15:14:53Z,a16e47f5-aa54-47fe-87e4-bb8af91a9fdd,Ennio Morricone,0b04407b-f517-4e00-9e6a-494795efc73e,Ninna Nanna In Blu (Raw Deal Remix),2009-05-03 15:14:53+00:00,15,afternoon,6,weekend
17,user_000001,2009-05-03T15:10:18Z,463a94f1-2713-40b1-9c88-dcc9c0170cae,Minus 8,4e78efc4-e545-47af-9617-05ff816d86e2,Elysian Fields,2009-05-03 15:10:18+00:00,15,afternoon,6,weekend
18,user_000001,2009-05-03T15:04:31Z,ad0811ea-e213-451d-b22f-fa1a7f9e0226,Beanfield,fb51d2c4-cc69-4128-92f5-77ec38d66859,Planetary Deadlock,2009-05-03 15:04:31+00:00,15,afternoon,6,weekend


##### Listening sessions
A session groups songs played close in time.
```
If gap between songs > 30 minutes → new session
```

In [143]:
# extract listening session

df_lastfm = df_lastfm.sort_values(['userid', 'datetime'])
df_lastfm["time_diff"] = df_lastfm.groupby("userid")["datetime"].diff().dt.seconds # in seconds

SESSION_GAP = 30 * 60 # in seconds

df_lastfm["new_session"] = (df_lastfm["time_diff"] > SESSION_GAP) | (df_lastfm["time_diff"].isna())

df_lastfm['session_id'] = df_lastfm.groupby("userid")["new_session"].cumsum()

df_lastfm.head(100)

,userid,timestamp,artid,artname,traid,traname,datetime,hour,time_of_day,day_of_week,day_type,time_diff,new_session,session_id
16684,user_000001,2006-08-13T13:59:20Z,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,c4633ab1-e715-477f-8685-afa5f2058e42,The Launching Of Big Face,2006-08-13 13:59:20+00:00,13,afternoon,6,weekend,NaN,True,1
16683,user_000001,2006-08-13T14:03:29Z,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,bc2765af-208c-44c5-b3b0-cf597a646660,Zn Zero,2006-08-13 14:03:29+00:00,14,afternoon,6,weekend,249.0,False,1
16682,user_000001,2006-08-13T14:10:43Z,09a114d9-7723-4e14-b524-379697f6d2b5,Plaid & Bob Jaroc,aa9c5a80-5cbe-42aa-a966-eb3cfa37d832,The Return Of Super Barrio - End Credits,2006-08-13 14:10:43+00:00,14,afternoon,6,weekend,434.0,False,1
16681,user_000001,2006-08-13T14:17:40Z,67fb65b5-6589-47f0-9371-8a40eb268dfb,Tommy Guerrero,d9b1c1da-7e47-4f97-a135-77260f2f559d,Mission Flats,2006-08-13 14:17:40+00:00,14,afternoon,6,weekend,417.0,False,1
16680,user_000001,2006-08-13T14:19:06Z,1cfbc7d1-299c-46e6-ba4c-1facb84ba435,Artful Dodger,120bb01c-03e4-465f-94a0-dce5e9fac711,What You Gonna Do?,2006-08-13 14:19:06+00:00,14,afternoon,6,weekend,86.0,False,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16589,user_000001,2006-08-16T11:10:38Z,07729ca8-b15b-44d4-881b-6a325f61665a,Maysa,d1bdae20-9ba3-4c49-ae3c-99089cdd6d1a,Blue Horizon,2006-08-16 11:10:38+00:00,11,morning,2,weekday,62.0,False,3
16588,user_000001,2006-08-16T11:14:54Z,07729ca8-b15b-44d4-881b-6a325f61665a,Maysa,bbb904d2-d28b-4ecf-933a-6aa6720d71de,Osaka,2006-08-16 11:14:54+00:00,11,morning,2,weekday,256.0,False,3
16587,user_000001,2006-08-16T11:16:34Z,07729ca8-b15b-44d4-881b-6a325f61665a,Maysa,b37a264d-4f4d-4289-b2c4-8a9dc102171d,Everything,2006-08-16 11:16:34+00:00,11,morning,2,weekday,100.0,False,3
16586,user_000001,2006-08-16T13:43:02Z,07729ca8-b15b-44d4-881b-6a325f61665a,Maysa,4d201e6a-e823-4ed0-80a3-ebc32492c369,Tailor Made Love,2006-08-16 13:43:02+00:00,13,afternoon,2,weekday,8788.0,True,4


##### User taste
User taste means what artists or songs the user prefers.
```
play_count
```

In [144]:
# extract user prefer artist
user_artist_pref = (
    df_lastfm.groupby(["userid", "artname"])
      .size()
      .reset_index(name="play_count")
)
user_artist_pref.sort_values("play_count", ascending=False) # top artist

,userid,artname,play_count
7999,user_000008,Kanye West,26496
35909,user_000033,Dir En Grey,12005
36462,user_000033,ムック,7788
36555,user_000033,雅-Miyavi-,6018
8047,user_000008,T.I.,5314
...,...,...,...
43520,user_000041,Xyz,1
43519,user_000041,Xstormer1,1
43516,user_000041,X-Noize,1
43,user_000001,Babylonwaves.Com,1


#### Feature extraction summary
From the original columns you can derive:

From `timestamp`

- time_of_day
- weekday/weekend
- hour
- month
- season

From `timestamp gaps`

- session_id
- session_length
- session_position

From `listening history`

- user favorite artists
- user favorite songs
- artist popularity

#### User Favorite Song
Normalized play count per user per song.
```
song_preference = play_count of song / total play_count of user
```

In [145]:
#User Favorite Song
user_song_pref = (df_lastfm.groupby(["userid", "traname"]).size().reset_index(name="play_count"))

# total plays per user
user_total = df_lastfm.groupby("userid").size().reset_index(name="total_plays")

# merge and normalize
user_song_pref = user_song_pref.merge(user_total, on="userid")
user_song_pref["song_preference"] = user_song_pref["play_count"] / user_song_pref["total_plays"]

user_song_pref.sort_values("song_preference", ascending=False).head(10)

,userid,traname,play_count,total_plays,song_preference
36279,user_000010,Omen,377,6574,0.057347
32769,user_000008,Heartless,2119,37235,0.056909
33005,user_000008,See You In My Nightmares,2069,37235,0.055566
32999,user_000008,Say You Will,2065,37235,0.055459
32861,user_000008,Love Lockdown,2059,37235,0.055297
33130,user_000008,Welcome To Heartbreak (Feat. Kid Cudi),2059,37235,0.055297
32650,user_000008,Coldest Winter,2052,37235,0.055109
32942,user_000008,Paranoid (Feat. Mr. Hudson),2051,37235,0.055083
32587,user_000008,Amazing (Feat. Young Jeezy),2049,37235,0.055029
32948,user_000008,Pinocchio Story (Freestyle Live From Singapore),2041,37235,0.054814


#### User Popurality
Global popurality based on unique listener count, normalized to 0-1
```
artist_popurality = unique_listener / max(unique_listeners)
```

In [146]:
artist_pop = (
    df_lastfm.groupby("artname")["userid"]
      .nunique()
      .reset_index(name="unique_listeners")
)

# normalize to 0-1
artist_pop["popularity"] = artist_pop["unique_listeners"] / artist_pop["unique_listeners"].max()

artist_pop.sort_values("popularity", ascending=False).head(10)

,artname,unique_listeners,popularity
4011,Coldplay,31,1.000000
15497,Radiohead,30,0.967742
571,Air,29,0.935484
20404,The White Stripes,29,0.935484
7802,Gorillaz,29,0.935484
13880,Nine Inch Nails,28,0.903226
4745,David Bowie,28,0.903226
13381,Muse,27,0.870968
15010,Placebo,27,0.870968
11894,Madonna,27,0.870968


## Recommendation Engine

#### Merge both dataset

In [147]:

def normalise(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9 ]", "", s)   # strip punctuation
    s = re.sub(r"\s+", " ", s)
    return s

df_spotify['name_norm']    = df_spotify['name'].apply(normalise)
df_spotify['artists_norm'] = df_spotify['artists'].apply(normalise)

df_lastfm['traname_norm'] = df_lastfm['traname'].apply(normalise)
df_lastfm['artname_norm'] = df_lastfm['artname'].apply(normalise)

# exact merge on both columns
merged = df_lastfm.merge(
    df_spotify,
    left_on=['artname_norm', 'traname_norm'],
    right_on=['artists_norm', 'name_norm'],
    how='left'
)

match_rate = merged['tempo'].notna().mean()
print(f"Match rate: {match_rate:.1%}")

Match rate: 71.8%


In [148]:
merged.columns

Index(['userid', 'timestamp', 'artid', 'artname', 'traid', 'traname',
       'datetime', 'hour', 'time_of_day', 'day_of_week', 'day_type',
       'time_diff', 'new_session', 'session_id', 'traname_norm',
       'artname_norm', 'valence', 'year', 'acousticness', 'artists',
       'danceability', 'duration_ms', 'energy', 'explicit', 'id',
       'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'name',
       'popularity', 'release_date', 'speechiness', 'tempo', 'name_norm',
       'artists_norm'],
      dtype='str')

In [149]:
merged['liveness'].notna().sum()

np.int64(2039665)

In [150]:
merged[merged['liveness'].notna()].head()

,userid,timestamp,artid,artname,traid,traname,datetime,hour,time_of_day,day_of_week,...,liveness,loudness,mode,name,popularity,release_date,speechiness,tempo,name_norm,artists_norm
8,user_000001,2006-08-13T14:59:59Z,a74b1b7f-71a5-4011-9441-d0b5e4122711,Radiohead,1c0377bb-c00b-4bbe-b4b2-615f13324adc,Idioteque,2006-08-13 14:59:59+00:00,14,afternoon,6,...,0.0914,-7.800,1.0,Idioteque,59.0,2000-10-01,0.2400,137.544,idioteque,radiohead
17,user_000001,2006-08-13T15:44:17Z,69158f97-4c07-4c4e-baf8-4e4ab1ed666e,Boards Of Canada,ef756a43-0905-4e59-8f05-a6b2bd977aa8,Dayvan Cowboy,2006-08-13 15:44:17+00:00,15,afternoon,6,...,0.0677,-10.774,1.0,Dayvan Cowboy,55.0,2005-10-17,0.0427,167.950,dayvan cowboy,boards of canada
25,user_000001,2006-08-13T16:24:47Z,a74b1b7f-71a5-4011-9441-d0b5e4122711,Radiohead,31c89fdc-afdb-4d84-bc61-2dbabf0b4f23,Treefingers,2006-08-13 16:24:47+00:00,16,afternoon,6,...,0.1090,-21.359,1.0,Treefingers,63.0,2000-10-01,0.0354,138.305,treefingers,radiohead
33,user_000001,2006-08-13T16:57:01Z,69158f97-4c07-4c4e-baf8-4e4ab1ed666e,Boards Of Canada,93fa1eb2-dd7c-4e0e-8174-360d32176586,Peacock Tail,2006-08-13 16:57:01+00:00,16,afternoon,6,...,0.0788,-13.083,1.0,Peacock Tail,52.0,2005-10-17,0.0580,171.424,peacock tail,boards of canada
36,user_000001,2006-08-13T17:09:53Z,69158f97-4c07-4c4e-baf8-4e4ab1ed666e,Boards Of Canada,ef756a43-0905-4e59-8f05-a6b2bd977aa8,Dayvan Cowboy,2006-08-13 17:09:53+00:00,17,evening,6,...,0.0677,-10.774,1.0,Dayvan Cowboy,55.0,2005-10-17,0.0427,167.950,dayvan cowboy,boards of canada


###  Build the track item matrix
one row per unique (artname, traname) with context features

In [151]:
AUDIO_FEATURES = [
    'acousticness', 'danceability', 'energy',
    'instrumentalness', 'loudness', 'speechiness',
    'tempo', 'valence', 'popularity'
]
TIME_SLOTS = ['morning', 'afternoon', 'evening', 'night']

# unique tracks enriched with spotify features
tracks = merged.groupby(['artname', 'traname']).agg(
    total_plays=('userid', 'count'),
    **{f: (f, 'mean') for f in AUDIO_FEATURES}   # avg if multiple rows
).reset_index()

# --- fill missing audio features with artist-level mean ---
for feat in AUDIO_FEATURES:
    artist_means = tracks.groupby('artname')[feat].transform('mean')
    tracks[feat] = tracks[feat].fillna(artist_means)
    tracks[feat] = tracks[feat].fillna(tracks[feat].median())  # global fallback

# --- scale audio features to [0, 1] ---
scaler = MinMaxScaler()
tracks[AUDIO_FEATURES] = scaler.fit_transform(tracks[AUDIO_FEATURES])

# --- time-of-day distribution from Last.fm ---
tod_counts = (
    merged.groupby(['artname', 'traname', 'time_of_day'])
    .size().unstack(fill_value=0)
    .reindex(columns=TIME_SLOTS, fill_value=0)
)
tod_counts = tod_counts.div(tod_counts.sum(axis=1).replace(0, 1), axis=0)
tracks = tracks.merge(tod_counts.reset_index(), on=['artname', 'traname'], how='left')
tracks[TIME_SLOTS] = tracks[TIME_SLOTS].fillna(0.25)

tracks['track_id'] = range(len(tracks))

# --- final feature matrix: audio + time-of-day ---
from scipy.sparse import csr_matrix
audio_matrix = csr_matrix(tracks[AUDIO_FEATURES].values)
tod_matrix   = csr_matrix(tracks[TIME_SLOTS].values)
item_matrix  = hstack([audio_matrix, tod_matrix])

In [152]:
tracks

,artname,traname,total_plays,acousticness,danceability,energy,instrumentalness,loudness,speechiness,tempo,valence,popularity,morning,afternoon,evening,night,track_id
0,! Www.Polskie-Mp3.Tk ! Katarzyna Groniec,02. Tango Z Pistoletem,1,0.132530,0.528571,0.693694,0.028140,0.863955,0.051249,0.545637,0.510101,0.539377,0.000000,0.000000,1.000000,0.000000,0
1,!!!,A New Name,11,0.132530,0.528571,0.693694,0.028140,0.863955,0.051249,0.545637,0.510101,0.539377,0.181818,0.454545,0.090909,0.272727,1
2,!!!,All My Heroes Are Weirdos,17,0.132530,0.528571,0.693694,0.028140,0.863955,0.051249,0.545637,0.510101,0.539377,0.176471,0.411765,0.117647,0.294118,2
3,!!!,Bend Over Beethoven,10,0.132530,0.528571,0.693694,0.028140,0.863955,0.051249,0.545637,0.510101,0.539377,0.000000,0.600000,0.400000,0.000000,3
4,!!!,Break In Case Of Anything,10,0.132530,0.528571,0.693694,0.028140,0.863955,0.051249,0.545637,0.510101,0.539377,0.200000,0.600000,0.200000,0.000000,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158842,소녀시대,Kissing You,1,0.799576,0.458823,0.247366,0.121283,0.747327,0.046534,0.491949,0.452694,0.213584,0.000000,0.000000,0.000000,1.000000,158842
158843,소녀시대,Ooh La-La!,2,0.799576,0.458823,0.247366,0.121283,0.747327,0.046534,0.491949,0.452694,0.213584,0.000000,0.000000,0.000000,1.000000,158843
158844,소녀시대,소녀시대,407,0.799576,0.458823,0.247366,0.121283,0.747327,0.046534,0.491949,0.452694,0.213584,0.000000,0.000000,0.000000,1.000000,158844
158845,Ｋｃｅｊ,Ｍａｉｎ Ｔｈｅｍｅ,814,0.799576,0.458823,0.247366,0.121283,0.747327,0.046534,0.491949,0.452694,0.213584,0.000000,0.000000,0.500000,0.500000,158845


### Build the user profile vector
weighted avg of all tracks the user played -> one vector per user


In [153]:
def build_user_profile(userid, df, tracks, item_matrix):
    user_plays = (
        df[df['userid'] == userid]
        .groupby(['artname', 'traname'])
        .size().reset_index(name='play_count')
    )
    user_plays = user_plays.merge(
        tracks[['artname', 'traname', 'track_id']],
        on=['artname', 'traname'], how='inner'
    )
    if user_plays.empty:
        return None, None

    weights = user_plays['play_count'].values.astype(float)
    weights /= weights.sum()
    profile = csr_matrix(weights) @ item_matrix[user_plays['track_id'].values]
    return profile, set(zip(user_plays['artname'], user_plays['traname']))

### Candidate pool
track that the user has **not** heard

In [154]:
def get_candidates(userid, df, tracks):
    heard = set(zip(
        df[df['userid'] == userid]['artname'],
        df[df['userid'] == userid]['traname']
    ))
    candidates = tracks[~tracks.apply(
        lambda r: (r['artname'], r['traname']) in heard, axis=1
    )]
    return candidates

### Cosine similarity
score every candidate against the user profile vector


In [155]:
from sklearn.metrics.pairwise import cosine_similarity

def score_candidates(user_profile, candidates, item_matrix):
    cand_vectors = item_matrix[candidates['track_id'].values]
    scores = cosine_similarity(user_profile, cand_vectors).flatten()
    candidates = candidates.copy()
    candidates['base_score'] = scores
    return candidates

### Context boost
multiply score if track's typical time_of_day matches current session


In [156]:
BOOST = 1.25  # 25% lift

def apply_context_boost(candidates, current_time_of_day):
    candidates = candidates.copy()
    # dominant TOD for each candidate track
    candidates['dominant_tod'] = candidates[TIME_SLOTS].idxmax(axis=1)
    match = candidates['dominant_tod'] == current_time_of_day
    candidates['final_score'] = candidates['base_score'] * np.where(match, BOOST, 1.0)
    return candidates

### Rank and return top n
sort by boosted score, return top 5–10 with explanation string


In [157]:
def recommend(userid, current_time_of_day, df, tracks, item_matrix, n=10):
    profile, _ = build_user_profile(userid, df, tracks, item_matrix)
    if profile is None:
        return pd.DataFrame()

    candidates = get_candidates(userid, df, tracks)
    candidates = score_candidates(profile, candidates, item_matrix)
    candidates = apply_context_boost(candidates, current_time_of_day)
    top_n = candidates.sort_values('final_score', ascending=False).head(n)
    return top_n

### Rule base explain
generate "why" string from the top contributing feature


In [158]:
# Settings to use
USER = 'user_000001'
CURRENT_TIME_OF_DAY = 'night'

FEATURE_LABELS = {
    'energy':          'high energy',
    'acousticness':    'acoustic feel',
    'danceability':    'danceability',
    'valence':         'positive mood',
    'instrumentalness':'instrumental style',
    'tempo':           'tempo',
}

In [159]:
# define explain() to use correct signature
def explain(row, user_means, current_time_of_day):
    diffs = {f: abs(row[f] - user_means[f]) for f in FEATURE_LABELS}
    top_feature = min(diffs, key=diffs.get)
    label = FEATURE_LABELS[top_feature]

    parts = [f"Matches your preferred {label}"]
    if row['dominant_tod'] == current_time_of_day:
        parts.append(f"fits your {current_time_of_day} listening pattern")
    if row['total_plays'] > tracks['total_plays'].quantile(0.75):
        parts.append("popular track overall")
    return " · ".join(parts)

In [160]:
# Step 1 — compute user's mean for each audio feature (do this ONCE, outside apply)
user_heard = df_lastfm[df_lastfm['userid'] == USER][['artname', 'traname']]
user_tracks = user_heard.merge(tracks, on=['artname', 'traname'], how='inner')

# guard agaist user_tracks empty
if user_tracks.empty or user_tracks[list(FEATURE_LABELS)].isnull().all().all():
    print(f"No enriched tracks found for user — check merge step")
else:
    user_means = {f: user_tracks[f].mean() for f in FEATURE_LABELS}
# user_means is now e.g. {'energy': 0.62, 'acousticness': 0.18, ...}

# Step 2 — call recommend & explain
results = recommend(USER, CURRENT_TIME_OF_DAY, df_lastfm, tracks, item_matrix)
results['explanation'] = results.apply(
    lambda r: explain(r, user_means, CURRENT_TIME_OF_DAY), axis=1
)

print(results[['artname', 'traname', 'final_score', 'explanation']].to_string())

                             artname                                     traname  final_score                                                                                       explanation
121036                  Sweet Coffee  Keep On Running (Sterac Electronics Remix)     1.193682  Matches your preferred positive mood · fits your night listening pattern · popular track overall
97960                  Peter Doherty                             Stix And Stones     1.193560  Matches your preferred positive mood · fits your night listening pattern · popular track overall
9669    Asle Bjorn Pr Leya Ft Anne K                     Lucky You__Original Mix     1.193015  Matches your preferred positive mood · fits your night listening pattern · popular track overall
30501                          Dalgo                                    Naw Mean     1.192854  Matches your preferred positive mood · fits your night listening pattern · popular track overall
53116                   Green On Red    

In [161]:
user_heard

,artname,traname
16684,Plaid & Bob Jaroc,The Launching Of Big Face
16683,Plaid & Bob Jaroc,Zn Zero
16682,Plaid & Bob Jaroc,The Return Of Super Barrio - End Credits
16681,Tommy Guerrero,Mission Flats
16680,Artful Dodger,What You Gonna Do?
...,...,...
4,坂本龍一,Mc1 (Live_2009_4_15)
3,坂本龍一,Hibari (Live_2009_4_15)
2,坂本龍一,Mc2 (Live_2009_4_15)
1,坂本龍一,Composition 0919 (Live_2009_4_15)


In [162]:
results

,artname,traname,total_plays,acousticness,danceability,energy,instrumentalness,loudness,speechiness,tempo,...,popularity,morning,afternoon,evening,night,track_id,base_score,dominant_tod,final_score,explanation
121036,Sweet Coffee,Keep On Running (Sterac Electronics Remix),75,0.13253,0.528571,0.693694,0.02814,0.863955,0.051249,0.545637,...,0.539377,0.173333,0.320000,0.173333,0.333333,121036,0.954946,night,1.193682,Matches your preferred positive mood · fits yo...
97960,Peter Doherty,Stix And Stones,29,0.13253,0.528571,0.693694,0.02814,0.863955,0.051249,0.545637,...,0.539377,0.137931,0.344828,0.137931,0.379310,97960,0.954848,night,1.193560,Matches your preferred positive mood · fits yo...
9669,Asle Bjorn Pr Leya Ft Anne K,Lucky You__Original Mix,53,0.13253,0.528571,0.693694,0.02814,0.863955,0.051249,0.545637,...,0.539377,0.150943,0.339623,0.132075,0.377358,9669,0.954412,night,1.193015,Matches your preferred positive mood · fits yo...
30501,Dalgo,Naw Mean,33,0.13253,0.528571,0.693694,0.02814,0.863955,0.051249,0.545637,...,0.539377,0.212121,0.333333,0.090909,0.363636,30501,0.954283,night,1.192854,Matches your preferred positive mood · fits yo...
53116,Green On Red,Time Ain'T Nothing,53,0.13253,0.528571,0.693694,0.02814,0.863955,0.051249,0.545637,...,0.539377,0.150943,0.320755,0.188679,0.339623,53116,0.954189,night,1.192736,Matches your preferred positive mood · fits yo...
94957,Pain Of Salvation,Dae Pecuniae,34,0.13253,0.528571,0.693694,0.02814,0.863955,0.051249,0.545637,...,0.539377,0.176471,0.323529,0.147059,0.352941,94957,0.954187,night,1.192734,Matches your preferred positive mood · fits yo...
12,!!!,Hello? Is This Thing On?,27,0.13253,0.528571,0.693694,0.02814,0.863955,0.051249,0.545637,...,0.539377,0.074074,0.370370,0.148148,0.407407,12,0.954120,night,1.192650,Matches your preferred positive mood · fits yo...
75167,Logic,Fucking Swedish,43,0.13253,0.528571,0.693694,0.02814,0.863955,0.051249,0.545637,...,0.539377,0.139535,0.325581,0.186047,0.348837,75167,0.954119,night,1.192649,Matches your preferred positive mood · fits yo...
65833,Joy Zipper,For Lenny'S Own Pleasure,31,0.13253,0.528571,0.693694,0.02814,0.863955,0.051249,0.545637,...,0.539377,0.193548,0.322581,0.129032,0.354839,65833,0.953866,night,1.192332,Matches your preferred positive mood · fits yo...
37845,Dog Years,In Stitches,21,0.13253,0.528571,0.693694,0.02814,0.863955,0.051249,0.545637,...,0.539377,0.095238,0.380952,0.095238,0.428571,37845,0.953830,night,1.192288,Matches your preferred positive mood · fits yo...


### Collaborative Filter

In [163]:
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

# User-item play count matrix
user_item = (
    df_lastfm.groupby(['userid', 'traname'])
    .size()
    .reset_index(name='play_count')
)

# Pivot to matrix
ui_pivot = user_item.pivot(index='userid', columns='traname', values='play_count').fillna(0)

# Convert to sparse for efficiency
ui_sparse = csr_matrix(ui_pivot.values)

user_index = {u: i for i, u in enumerate(ui_pivot.index)}
track_index = {t: i for i, t in enumerate(ui_pivot.columns)}

In [164]:
# Latent factor model — 50 components is a good starting point
svd = TruncatedSVD(n_components=50, random_state=42)
user_factors = svd.fit_transform(ui_sparse)        # shape: (n_users, 50)
item_factors = svd.components_.T                   # shape: (n_tracks, 50)

# Reconstruct predicted ratings
predicted = user_factors @ item_factors.T           # shape: (n_users, n_tracks)

In [165]:
def recommend_cf(userid, df, n=10):
    if userid not in user_index:
        return pd.DataFrame()   # cold-start: no CF signal

    u_idx = user_index[userid]
    scores = predicted[u_idx]                      # score for every track

    # Mask already-heard tracks
    heard_tracks = set(df[df['userid'] == userid]['traname'])
    results = []
    for track, t_idx in track_index.items():
        if track not in heard_tracks:
            results.append({'traname': track, 'cf_score': scores[t_idx]})

    return pd.DataFrame(results).sort_values('cf_score', ascending=False).head(n)

In [166]:
ALPHA = 0.6   # weight for content-based score
BETA  = 0.4   # weight for CF score

def recommend_hybrid(userid, current_time_of_day, df, tracks, item_matrix, n=10):
    # --- Content-based scores (your existing pipeline) ---
    cb_results = recommend(userid, current_time_of_day, df, tracks, item_matrix, n=200)
    if cb_results.empty:
        return pd.DataFrame()

    # Normalise content scores to [0, 1]
    cb_results['cb_norm'] = (
        cb_results['final_score'] / cb_results['final_score'].max()
    )

    # --- CF scores ---
    if userid in user_index:
        u_idx = user_index[userid]
        cf_scores = predicted[u_idx]
        cb_results['cf_norm'] = cb_results['traname'].map(
            lambda t: cf_scores[track_index[t]] if t in track_index else 0.0
        )
        # Normalise CF scores
        max_cf = cb_results['cf_norm'].max()
        if max_cf > 0:
            cb_results['cf_norm'] /= max_cf
    else:
        cb_results['cf_norm'] = 0.0   # cold-start fallback

    # --- Blend ---
    cb_results['hybrid_score'] = ALPHA * cb_results['cb_norm'] + BETA * cb_results['cf_norm']

    return cb_results.sort_values('hybrid_score', ascending=False).head(n)

In [167]:
USER = 'user_000001'
CURRENT_TIME_OF_DAY = 'night'

results = recommend_hybrid(USER, CURRENT_TIME_OF_DAY, df_lastfm, tracks, item_matrix)
print(results[['artname', 'traname', 'hybrid_score']].to_string())

                             artname                                     traname  hybrid_score
45894                        Fembots                                    The City      0.995569
19291                  Brand X Music                                    Fearless      0.795452
135294                      The Owls                                         Air      0.729340
121036                  Sweet Coffee  Keep On Running (Sterac Electronics Remix)      0.600000
97960                  Peter Doherty                             Stix And Stones      0.599939
9669    Asle Bjorn Pr Leya Ft Anne K                     Lucky You__Original Mix      0.599665
30501                          Dalgo                                    Naw Mean      0.599584
53116                   Green On Red                          Time Ain'T Nothing      0.599524
94957              Pain Of Salvation                                Dae Pecuniae      0.599523
12                               !!!              

---
## Layer 2 — SHAP  ·  Layer 3 — LIME
These cells run directly after `recommend_hybrid()`.  
All variables (`tracks`, `item_matrix`, `AUDIO_FEATURES`, `TIME_SLOTS`,
`FEATURE_LABELS`, `USER`, `CURRENT_TIME_OF_DAY`) are inherited from the cells above.

### Shared scorer + background
Both SHAP and LIME need:
- `scorer(X)` — wraps cosine similarity vs the user profile (same maths as `score_candidates`)
- `background` — 100 tracks the user has already heard, used as the reference baseline

In [168]:
from sklearn.metrics.pairwise import cosine_similarity as _cos

ALL_FEATURES = AUDIO_FEATURES + TIME_SLOTS   # 13 features in item_matrix order

# --- dense user profile (same weighted avg as build_user_profile) ---------
def _profile_dense(userid):
    profile_sparse, _ = build_user_profile(userid, df_lastfm, tracks, item_matrix)
    if profile_sparse is None:
        raise ValueError(f'No history for {userid}')
    return profile_sparse.toarray().flatten()   # shape (13,)

# --- scorer: f(X [n,13]) -> scores [n] ------------------------------------
def _make_scorer(profile_1d):
    p = profile_1d.reshape(1, -1)
    return lambda X: _cos(p, X).flatten()

# --- background: heard tracks as dense numpy array ------------------------
def _background(userid, n=100):
    heard = set(zip(df_lastfm[df_lastfm['userid']==userid]['artname'],
                    df_lastfm[df_lastfm['userid']==userid]['traname']))
    ht = tracks[tracks.apply(lambda r: (r['artname'], r['traname']) in heard, axis=1)]
    mat = ht[ALL_FEATURES].values.astype(float)
    if len(mat) > n:
        mat = mat[np.random.choice(len(mat), n, replace=False)]
    return mat

# Pre-compute once for USER
profile_dense = _profile_dense(USER)
scorer        = _make_scorer(profile_dense)
background    = _background(USER, n=100)
print(f'Profile shape: {profile_dense.shape}  |  Background: {background.shape}')

Profile shape: (13,)  |  Background: (100, 13)


### SHAP — feature contribution scores
`KernelExplainer` perturbs each track's feature vector and measures
how much each feature pushes the cosine-similarity score **up (+) or down (−)**
relative to the user's baseline (their heard-tracks average).

In [169]:
# Build explainer (runs once; ~10-30 s for 100 background samples)
print('Building SHAP explainer …')
shap_explainer = shap.KernelExplainer(scorer, background)
print('Done.')

# Get hybrid recommendations
hybrid_results = recommend_hybrid(USER, CURRENT_TIME_OF_DAY, df_lastfm, tracks, item_matrix)

# Compute SHAP values for every recommendation
shap_rows = []
for _, row in hybrid_results.iterrows():
    x  = row[ALL_FEATURES].values.reshape(1, -1).astype(float)
    sv = shap_explainer.shap_values(x, silent=True)[0]   # shape (13,)
    entry = dict(zip(ALL_FEATURES, sv))
    entry.update({'artname': row['artname'], 'traname': row['traname'],
                  'hybrid_score': row['hybrid_score']})
    shap_rows.append(entry)

shap_df = pd.DataFrame(shap_rows)

# --- SHAP waterfall chart for the #1 recommendation ----------------------
top_row   = hybrid_results.iloc[0]
top_shap  = shap_df.iloc[0]
sv_vals   = np.array([top_shap[f] for f in ALL_FEATURES])
order     = np.argsort(np.abs(sv_vals))[::-1]
feat_ord  = [ALL_FEATURES[i] for i in order]
sv_ord    = sv_vals[order]

fig, ax = plt.subplots(figsize=(9, 5))
colors  = ['#2ecc71' if v > 0 else '#e74c3c' for v in sv_ord]
bars    = ax.barh(feat_ord[::-1], sv_ord[::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('SHAP value  (impact on cosine-similarity score)')
ax.set_title(f"SHAP — {top_row['artname']} · {top_row['traname']}")
for bar, val in zip(bars[::-1], sv_ord[::-1]):
    ax.text(val + (0.002 if val >= 0 else -0.002),
            bar.get_y() + bar.get_height() / 2,
            f'{val:+.4f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=8)
plt.tight_layout()
plt.show()

# --- SHAP explanation string (replaces old rule-based explain()) ----------
def shap_explain_string(track_row, sv_dict, tod, top_n=3):
    audio_sv = {f: sv_dict[f] for f in AUDIO_FEATURES}
    top_pos  = sorted([(f,v) for f,v in audio_sv.items() if v > 0],
                      key=lambda x: x[1], reverse=True)[:top_n]
    if top_pos:
        feat_str = ', '.join(f"{FEATURE_LABELS.get(f,f)} ({v:+.3f})" for f,v in top_pos)
        parts = [f'Strong match on {feat_str}']
    else:
        parts = ['Broad catalogue match']
    if track_row.get('dominant_tod') == tod:
        parts.append(f'fits your {tod} pattern')
    if track_row.get('total_plays', 0) > tracks['total_plays'].quantile(0.75):
        parts.append('popular overall')
    return ' · '.join(parts)

hybrid_results = hybrid_results.copy()
for i, row in hybrid_results.iterrows():
    sv_dict = shap_df[shap_df['traname'] == row['traname']].iloc[0].to_dict()
    hybrid_results.loc[i, 'shap_explanation'] = shap_explain_string(
        row, sv_dict, CURRENT_TIME_OF_DAY)

print('\nSHAP explanations:')
print(hybrid_results[['artname', 'traname', 'hybrid_score', 'shap_explanation']].to_string())

Building SHAP explainer …
Done.

SHAP explanations:
                             artname                                     traname  hybrid_score                                                                                                                       shap_explanation
45894                        Fembots                                    The City      0.995569  Strong match on acoustic feel (+0.006), instrumental style (+0.005), high energy (+0.005) · fits your night pattern · popular overall
19291                  Brand X Music                                    Fearless      0.795452  Strong match on acoustic feel (+0.006), instrumental style (+0.005), high energy (+0.004) · fits your night pattern · popular overall
135294                      The Owls                                         Air      0.729340  Strong match on acoustic feel (+0.006), instrumental style (+0.005), high energy (+0.004) · fits your night pattern · popular overall
121036                  Swee

### LIME — cross-validation of SHAP
`LimeTabularExplainer` fits a local linear model around each track.
Its coefficients independently confirm (or dispute) SHAP's top features.
Features both agree on are presented as **confident**.

In [ ]:
# Build LIME explainer (reuses the same background as SHAP)
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data         = background,
    feature_names         = ALL_FEATURES,
    mode                  = 'regression',
    discretize_continuous = False,
    random_state          = 42
)

def _lime_coefs(track_row):
    x   = track_row[ALL_FEATURES].values.astype(float)
    exp = lime_explainer.explain_instance(
        data_row     = x,
        predict_fn   = scorer,
        num_features = len(ALL_FEATURES),
        num_samples  = 500
    )
    coefs = {}
    for label, coef in exp.as_list():
        for feat in ALL_FEATURES:
            if feat in label:
                coefs[feat] = coef
                break
    return coefs

# Run LIME on every recommendation
print('Running LIME explanations …')
lime_rows = []
for _, row in hybrid_results.iterrows():
    lv = _lime_coefs(row)
    lv.update({'artname': row['artname'], 'traname': row['traname']})
    lime_rows.append(lv)
lime_df = pd.DataFrame(lime_rows)
print('Done.')

# --- LIME bar chart for the #1 recommendation ----------------------------
top_lime = lime_df.iloc[0]
lv_vals  = np.array([top_lime.get(f, 0) for f in ALL_FEATURES])
l_order  = np.argsort(np.abs(lv_vals))[::-1]
l_feats  = [ALL_FEATURES[i] for i in l_order]
l_vals   = lv_vals[l_order]

fig2, ax2 = plt.subplots(figsize=(9, 5))
lcolors = ['#3498db' if v > 0 else '#e67e22' for v in l_vals]
ax2.barh(l_feats[::-1], l_vals[::-1], color=lcolors[::-1])
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_xlabel('LIME coefficient  (local linear approximation)')
ax2.set_title(f"LIME — {top_row['artname']} · {top_row['traname']}")
plt.tight_layout()
plt.show()

# --- SHAP vs LIME agreement + combined final explanation ------------------
def _top_pos(val_dict, n=3):
    return [f for f,v in sorted(val_dict.items(), key=lambda x: x[1], reverse=True)
            if v > 0][:n]

def combined_explanation(track_row, sv_dict, lv_dict, tod, top_n=3):
    audio_sv   = {f: sv_dict.get(f, 0) for f in AUDIO_FEATURES}
    audio_lv   = {f: lv_dict.get(f, 0) for f in AUDIO_FEATURES}
    shap_top   = _top_pos(audio_sv, top_n)
    lime_top   = _top_pos(audio_lv, top_n)
    confident  = [f for f in shap_top if f in lime_top]
    uncertain  = [f for f in shap_top if f not in lime_top]

    lines = []

    # Line 1 — main drivers (confirmed by both SHAP & LIME)
    if confident:
        labels = [FEATURE_LABELS.get(f, f) for f in confident]
        lines.append(f"Primary drivers: {', '.join(labels)}")

    # Line 2 — secondary drivers (SHAP only)
    if uncertain:
        labels = [FEATURE_LABELS.get(f, f) for f in uncertain]
        lines.append(f"Supporting factor (SHAP only): {', '.join(labels)}")

    # Fallback
    if not confident and not uncertain:
        lines.append("🎵 Broad catalogue match across multiple features")

    # Line 3 — context match
    if track_row.get('dominant_tod') == tod:
        lines.append(f"Fits your {tod} listening pattern")

    # Line 4 — popularity
    if track_row.get('total_plays', 0) > tracks['total_plays'].quantile(0.75):
        lines.append("Popular track overall")

    return '\n    '.join(lines)

print(f"\n{'='*70}")
print('  SHAP vs LIME agreement  +  final combined explanation')
print(f"{'='*70}\n")

for (_, res_row), shap_row, lime_row in zip(
        hybrid_results.iterrows(), shap_rows, lime_rows):

    shap_top3 = _top_pos({f: shap_row.get(f,0) for f in AUDIO_FEATURES})
    lime_top3 = _top_pos({f: lime_row.get(f,0) for f in AUDIO_FEATURES})
    agreed    = sorted(set(shap_top3) & set(lime_top3))
    disputed  = sorted((set(shap_top3) | set(lime_top3)) - set(agreed))
    final_exp = combined_explanation(
        res_row, shap_row, lime_row, CURRENT_TIME_OF_DAY)

    print(f"{'*'*50}\n")
    print(f"  {res_row['artname']} — {res_row['traname']}")
    print(f"    Score      : {res_row['hybrid_score']:.4f}")
    print(f"    SHAP top 3 : {shap_top3}")
    print(f"    LIME top 3 : {lime_top3}")
    if agreed:   print(f"    ✓ Agree    : {agreed}")
    if disputed: print(f"    ✗ Dispute  : {disputed}")
    print(f"{final_exp}")
    print(f"{'*'*50}\n")

    print()

Running LIME explanations …
Done.

  SHAP vs LIME agreement  +  final combined explanation

  Fembots — The City
    Score      : 0.9956
    SHAP top 3 : ['acousticness', 'instrumentalness', 'energy']
    LIME top 3 : ['energy', 'instrumentalness', 'valence']
    ✓ Agree    : ['energy', 'instrumentalness']
    ✗ Dispute  : ['acousticness', 'valence']
    Final why  : Driven by instrumental style, high energy · Also acoustic feel (SHAP only) · fits your night listening pattern · popular track overall

  Brand X Music — Fearless
    Score      : 0.7955
    SHAP top 3 : ['acousticness', 'instrumentalness', 'energy']
    LIME top 3 : ['instrumentalness', 'acousticness', 'energy']
    ✓ Agree    : ['acousticness', 'energy', 'instrumentalness']
    Final why  : Driven by acoustic feel, instrumental style, high energy · fits your night listening pattern · popular track overall

  The Owls — Air
    Score      : 0.7293
    SHAP top 3 : ['acousticness', 'instrumentalness', 'energy']
    LIME to

### Evaluation Layer

In [171]:

from collections import defaultdict


def split_user_history(df, train_ratio=0.8):
    """Split each user's listening history chronologically."""
    train_list, test_list = [], []
    for uid, group in df.groupby('userid'):
        group = group.sort_values('datetime')
        n = len(group)
        split = int(n * train_ratio)
        if split < 5 or (n - split) < 1:
            continue  # skip users with too little data
        train_list.append(group.iloc[:split])
        test_list.append(group.iloc[split:])
    return pd.concat(train_list), pd.concat(test_list)

df_train, df_test = split_user_history(df_lastfm)
print(f"Train: {len(df_train):,} rows  |  Test: {len(df_test):,} rows")

# ground truth: set of (artname, traname) per user in test set
ground_truth = (
    df_test.groupby('userid')
    .apply(lambda g: set(zip(g['artname'], g['traname'])))
    .to_dict()
)

merged_train = df_train.merge(
    df_spotify,
    left_on=['artname_norm', 'traname_norm'],
    right_on=['artists_norm', 'name_norm'],
    how='left'
)

# Rebuild tracks from train
tracks_train = merged_train.groupby(['artname', 'traname']).agg(
    total_plays=('userid', 'count'),
    **{f: (f, 'mean') for f in AUDIO_FEATURES}
).reset_index()

for feat in AUDIO_FEATURES:
    artist_means = tracks_train.groupby('artname')[feat].transform('mean')
    tracks_train[feat] = tracks_train[feat].fillna(artist_means)
    tracks_train[feat] = tracks_train[feat].fillna(tracks_train[feat].median())

tracks_train[AUDIO_FEATURES] = scaler.fit_transform(tracks_train[AUDIO_FEATURES])

tod_counts_train = (
    merged_train.groupby(['artname', 'traname', 'time_of_day'])
    .size().unstack(fill_value=0)
    .reindex(columns=TIME_SLOTS, fill_value=0)
)
tod_counts_train = tod_counts_train.div(
    tod_counts_train.sum(axis=1).replace(0, 1), axis=0
)
tracks_train = tracks_train.merge(
    tod_counts_train.reset_index(), on=['artname', 'traname'], how='left'
)
tracks_train[TIME_SLOTS] = tracks_train[TIME_SLOTS].fillna(0.25)
tracks_train['track_id'] = range(len(tracks_train))

audio_matrix_tr = csr_matrix(tracks_train[AUDIO_FEATURES].values)
tod_matrix_tr   = csr_matrix(tracks_train[TIME_SLOTS].values)
item_matrix_tr  = hstack([audio_matrix_tr, tod_matrix_tr])

# Rebuild CF on train
user_item_train = (
    df_train.groupby(['userid', 'traname'])
    .size().reset_index(name='play_count')
)
ui_pivot_train = user_item_train.pivot(
    index='userid', columns='traname', values='play_count'
).fillna(0)
ui_sparse_train = csr_matrix(ui_pivot_train.values)

svd_train = TruncatedSVD(n_components=50, random_state=42)
user_factors_train = svd_train.fit_transform(ui_sparse_train)
item_factors_train = svd_train.components_.T
predicted_train    = user_factors_train @ item_factors_train.T

user_index_train = {u: i for i, u in enumerate(ui_pivot_train.index)}
track_index_train = {t: i for i, t in enumerate(ui_pivot_train.columns)}

print("Train models rebuilt.")

Train: 799,982 rows  |  Test: 200,018 rows
Train models rebuilt.


### Metric Functions

In [172]:
def precision_at_k(recommended, truth, k=10):
    """What fraction of top-K recommendations are in ground truth?"""
    rec_set = set(recommended[:k])
    if not rec_set:
        return 0.0
    return len(rec_set & truth) / len(rec_set)

def recall_at_k(recommended, truth, k=10):
    """What fraction of ground truth items appear in top-K?"""
    rec_set = set(recommended[:k])
    if not truth:
        return 0.0
    return len(rec_set & truth) / len(truth)

def ndcg_at_k(recommended, truth, k=10):
    """Normalized Discounted Cumulative Gain."""
    dcg = 0.0
    for i, item in enumerate(recommended[:k]):
        if item in truth:
            dcg += 1.0 / np.log2(i + 2)  # i+2 because rank starts at 1
    # ideal DCG: all hits at top positions
    n_relevant = min(len(truth), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(n_relevant))
    return dcg / idcg if idcg > 0 else 0.0

K = 10
eval_users = [u for u in ground_truth if len(ground_truth[u]) >= 3]
print(f"Evaluating on {len(eval_users)} users with K={K}\n")

results_eval = {'Content-Based': [], 'Collaborative': [], 'Hybrid': []}

for uid in eval_users:
    truth = ground_truth[uid]

    # --- Content-Based ---
    cb = recommend(uid, 'night', df_train, tracks_train, item_matrix_tr, n=K)
    if not cb.empty:
        cb_recs = list(zip(cb['artname'], cb['traname']))
    else:
        cb_recs = []

    # --- Collaborative Filtering ---
    cf_recs = []
    if uid in user_index_train:
        u_idx = user_index_train[uid]
        cf_scores = predicted_train[u_idx]
        heard = set(df_train[df_train['userid'] == uid]['traname'])
        cf_candidates = [
            (t, cf_scores[t_idx])
            for t, t_idx in track_index_train.items()
            if t not in heard
        ]
        cf_candidates.sort(key=lambda x: x[1], reverse=True)
        # need (artname, traname) tuples for matching
        for tname, _ in cf_candidates[:K]:
            match = tracks_train[tracks_train['traname'] == tname]
            if not match.empty:
                cf_recs.append((match.iloc[0]['artname'], tname))

    # --- Hybrid ---
    hy = recommend_hybrid(uid, 'night', df_train, tracks_train, item_matrix_tr, n=K)
    if not hy.empty:
        hy_recs = list(zip(hy['artname'], hy['traname']))
    else:
        hy_recs = []

    # Compute metrics
    for name, recs in [('Content-Based', cb_recs),
                        ('Collaborative', cf_recs),
                        ('Hybrid', hy_recs)]:
        results_eval[name].append({
            'userid': uid,
            'precision': precision_at_k(recs, truth, K),
            'recall':    recall_at_k(recs, truth, K),
            'ndcg':      ndcg_at_k(recs, truth, K),
        })

# Aggregate
print(f"{'Model':<20} {'Precision@10':>13} {'Recall@10':>11} {'NDCG@10':>9}")
print("-" * 56)
for model_name, records in results_eval.items():
    df_res = pd.DataFrame(records)
    p = df_res['precision'].mean()
    r = df_res['recall'].mean()
    n = df_res['ndcg'].mean()
    print(f"{model_name:<20} {p:>13.4f} {r:>11.4f} {n:>9.4f}")

    agree_count = 0
total_count = 0

for shap_row, lime_row in zip(shap_rows, lime_rows):
    shap_top3 = _top_pos({f: shap_row.get(f, 0) for f in AUDIO_FEATURES})
    lime_top3 = _top_pos({f: lime_row.get(f, 0) for f in AUDIO_FEATURES})
    overlap   = set(shap_top3) & set(lime_top3)
    union     = set(shap_top3) | set(lime_top3)
    if union:
        agree_count += len(overlap)
        total_count += len(union)

agreement_rate = agree_count / total_count if total_count > 0 else 0
print(f"\nSHAP vs LIME Agreement Rate: {agreement_rate:.1%}")
print(f"  (Jaccard overlap of top-3 positive features across {len(shap_rows)} recommendations)")

Evaluating on 41 users with K=10

Model                 Precision@10   Recall@10   NDCG@10
--------------------------------------------------------
Content-Based               0.0049      0.0000    0.0052
Collaborative               0.0000      0.0000    0.0000
Hybrid                      0.0366      0.0003    0.0468

SHAP vs LIME Agreement Rate: 57.9%
  (Jaccard overlap of top-3 positive features across 10 recommendations)
